# LSR — License-to-Operate & Sustainability Resilience

**6ᵉ KPI de gouvernance** du framework Human Capital Valuation CACEIS · Alberthon 2026

> Score sur 100 mesurant la **résilience sociale** de CACEIS face aux risques régulatoires, humains et réputationnels. Complète les 5 KPIs métier (HCVA · KTI · Skill Decay · RE-Score · SPE) en couvrant la dimension *Strategic fit / Savoir-être* du Deliverable 1.

```
LSR = (Mixité + Inclusion + Engagement) / 3
```

| Score | Question business | Calcul à partir de | Valeur 2024 |
|---|---|---|---|
| **Mixité** | *Sommes-nous équitables H/F ?* | Pay gap (Bilan Social) + % femmes encadrantes (Suivi accord) | **76 / 100** |
| **Inclusion** | *Les collaborateurs se sentent-ils inclus ?* | Baromètre D&I FR + Lux (pondéré effectif) | **67 / 100** |
| **Engagement** | *Vivent-ils CACEIS au-delà du contrat ?* | FAB'Life + We Care + Be Generous (intensité de participation) | **49 / 100** |

**LSR 2024 ≈ 64 / 100** · zone jaune

---

## Principe directeur

Chaque chiffre est **calculé par nous** à partir d'un tableau brut publié dans les documents CACEIS.

L'unique hypothèse explicite concerne le fallback des lignes `n/a` du fichier We Care : on remplit avec la **valeur minimale moyenne par type d'action** (= 24, moyenne du type *Ateliers*, le plus bas observé).

---

## Sommaire

1. Imports & configuration
2. Inputs sourcés (chiffres réels CACEIS 2024)
3. Composante Mixité (pay gap + % femmes encadrantes)
4. Composante Inclusion (Baromètre D&I FR + Lux)
5. Composante Engagement (intensité par employé)
6. LSR final + lecture board
7. Sensibilité
8. Visualisations
9. Export CSV (compatible Streamlit `app.py`)
10. Références

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CACEIS brand palette
STEEL = "#4A6E8C"
RED   = "#D65063"
BLUE  = "#5C768D"
GREY  = "#888B8D"
GREEN = "#2E8B57"
AMBER = "#E0A526"
DARK  = "#1F2937"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlecolor": STEEL,
})

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Inputs — données brutes verrouillées 2024

Toutes les valeurs sont extraites des documents CACEIS officiels. Source citée par ligne.

In [ ]:
# ============================================================
# Bilan Social 2024 — France
# Source: Bilan Social 2024.pdf, §2.1.2.2 "SAB moyen par catégorie x sexe"
# ============================================================
SAB_MOYEN_FEMMES_FR  = 54_753  # €
SAB_MOYEN_HOMMES_FR  = 60_360  # €

# ============================================================
# Suivi accord mixité diversité QVT 2024 — France
# Source: Suivi accord mixité 2024 vDef.pdf, "Promotion professionnelle" (~p.38)
# Tableau brut: Cadres encadrants par classification × sexe (31/12/2024)
# ============================================================
CADRES_ENCADRANTS_2024 = {
    # classification: (femmes, hommes)
    "H":  (0,   2),
    "I":  (10,  25),
    "J":  (21,  20),
    "K":  (25,  33),
    "HC": (41,  67),  # Hors classe
}

# ============================================================
# Baromètres Diversité & Inclusion 2025 — Mozaïk RH
# Sources:
#   - Baromètre D&I CACEIS - France.pdf, §2.6 "Sentiment d\'inclusion"
#   - Baromètre D&I CACEIS - Luxembourg.pdf, §2.6
# ============================================================
BAROMETRE_FR_PCT  = 70.0
BAROMETRE_LUX_PCT = 64.0

# Effectifs au 31/12/2024 — Bilans Sociaux 2024 FR + Lux
EFFECTIF_FR  = 2_043   # CDI + CDD (Bilan Social 2024.pdf §1.1.1)
EFFECTIF_LUX = 1_882   # CDI + CDD (Bilan Social 2024 Luxembourg.pdf §1.1.1)

# ============================================================
# Programmes sociaux 2024 — bilans annuels
# ============================================================

# --- FAB\'Life (santé, parentalité, bien-être, mois des diversités) ---
# Source: Bilan FAB\'Life 2024.pptx, slide 31 "FAB\'Life en quelques chiffres"
FAB_LIFE_PARTICIPATIONS_TOTAL  = 2_984   # incluant 743 du groupe Crédit Agricole SA
FAB_LIFE_PARTICIPATIONS_CACEIS = FAB_LIFE_PARTICIPATIONS_TOTAL - 743  # = 2 241

# --- We Care (santé au travail, D&I, parentalité, lutte discriminations, solidarité) ---
# Source: We Care - Bilan 2025.xlsx, sheet "Data" (49 actions)
WE_CARE_PARTICIPATIONS_CHIFFREES = 1_256   # somme des 35 lignes avec valeur numérique
WE_CARE_LIGNES_NA                = 14      # lignes "nc" ou "n/a" (Communications, E-learning, Charte...)

# Fallback pour les n/a : valeur minimale moyenne par type d\'action
# Moyennes par type de WE_CARE (sur les chiffrés uniquement) :
#   Ateliers         : 24   <- minimum  → règle de fallback
#   Webinaire/Conf   : 33,4
#   RDV individuels  : ~34
#   Action solidarité: 42,3
#   Spectacle        : 60
#   Intervention OS  : 88
WE_CARE_FALLBACK_PAR_NA = 24

# --- Be Generous (mécénat associatif des collaborateurs) ---
# Source: Bilan Groupe Be Generous CACEIS 2025.xlsx, sheet "Lauréats"
# Toutes entités CACEIS confondues (France, Luxembourg, Suisse, Pays-Bas, Allemagne)
BE_GENEROUS_LAUREATS_CACEIS  = 34
BE_GENEROUS_SUBVENTION_TOTAL = 84_680   # EUR

## 3. Composante Mixité

Score = moyenne de **deux sous-calculs**, chacun fait par nous depuis un tableau brut.

### 3.1 Pay gap — Bilan Social 2024 §2.1.2.2

```
gap (%) = (SAB_hommes − SAB_femmes) / SAB_hommes
score_paygap = max(0, 100 − |gap| × 5)     # 0% gap → 100, 20% gap → 0
```

### 3.2 % femmes au management — Suivi accord mixité 2024 (« Promotion professionnelle »)

Tableau brut des cadres encadrants au 31/12/2024 par classification × sexe (5 lignes, somme directe).

```
% femmes_encadrantes = total_femmes / total_encadrants
score_mgt = min(100, % / 40 × 100)         # cible légale parité = 40%
```

In [ ]:
# --- Pay gap ---
gap_pct = (SAB_MOYEN_HOMMES_FR - SAB_MOYEN_FEMMES_FR) / SAB_MOYEN_HOMMES_FR * 100
score_paygap = max(0, 100 - abs(gap_pct) * 5)

print(f"Pay gap H/F (SAB moyen)   : {gap_pct:.2f} %")
print(f"  → score_paygap          : {score_paygap:.1f} / 100")

# --- % femmes encadrantes ---
total_f_encadrants = sum(f for f, h in CADRES_ENCADRANTS_2024.values())
total_h_encadrants = sum(h for f, h in CADRES_ENCADRANTS_2024.values())
total_encadrants   = total_f_encadrants + total_h_encadrants
pct_femmes_mgt = total_f_encadrants / total_encadrants * 100
score_mgt = min(100, pct_femmes_mgt / 40 * 100)

print()
print(f"Cadres encadrants 31/12/2024 : {total_f_encadrants} F + {total_h_encadrants} H = {total_encadrants}")
print(f"  → % femmes encadrantes    : {pct_femmes_mgt:.2f} %")
print(f"  → score_mgt               : {score_mgt:.1f} / 100")

# --- Mixité ---
score_mixite = (score_paygap + score_mgt) / 2
print()
print(f"Mixité = (paygap + mgt) / 2 = ({score_paygap:.1f} + {score_mgt:.1f}) / 2 = {score_mixite:.1f}")

## 4. Composante Inclusion

Moyenne pondérée par effectif des Baromètres D&I 2025 publiés par Mozaïk RH (FR et Lux).

```
Inclusion = (FR × eff_FR + Lux × eff_Lux) / (eff_FR + eff_Lux)
```

In [ ]:
score_inclusion = (
    BAROMETRE_FR_PCT  * EFFECTIF_FR +
    BAROMETRE_LUX_PCT * EFFECTIF_LUX
) / (EFFECTIF_FR + EFFECTIF_LUX)

print(f"Inclusion FR : {BAROMETRE_FR_PCT} % × {EFFECTIF_FR} salariés")
print(f"Inclusion Lux: {BAROMETRE_LUX_PCT} % × {EFFECTIF_LUX} salariés")
print(f"  → Inclusion pondérée      : {score_inclusion:.1f} / 100")

## 5. Composante Engagement

**Intensité d'engagement** = participations totales aux programmes sociaux ÷ effectif.

Pas d'hypothèse sur l'unicité des employés (pas de tracking cross-programmes au niveau employé). Le ratio est un fait calculable, pas une estimation.

```
intensité = (FAB'Life + We Care + Be Generous) / effectif
score = min(100, intensité × 50)
```

Convention : 2 participations/employé/an = score 100, 1 participation = score 50.

**Fallback explicite** pour les 14 lignes `n/a` de We Care (Communications, E-learning, Charte…) : remplies avec **24 participants** par défaut, qui est la valeur minimale moyenne observée par type d'action (= moyenne du type *Ateliers*).

In [ ]:
# We Care avec fallback n/a
we_care_total = (
    WE_CARE_PARTICIPATIONS_CHIFFREES +
    WE_CARE_LIGNES_NA * WE_CARE_FALLBACK_PAR_NA
)
print(f"We Care chiffrés        : {WE_CARE_PARTICIPATIONS_CHIFFREES:>5}")
print(f"  + fallback n/a (×{WE_CARE_FALLBACK_PAR_NA}) : +{WE_CARE_LIGNES_NA * WE_CARE_FALLBACK_PAR_NA:>4}  ({WE_CARE_LIGNES_NA} lignes × {WE_CARE_FALLBACK_PAR_NA})")
print(f"  = We Care total estimé  : {we_care_total:>5}")

# Total participations CACEIS (FR + Lux principalement)
total_participations = (
    FAB_LIFE_PARTICIPATIONS_CACEIS +
    we_care_total +
    BE_GENEROUS_LAUREATS_CACEIS
)
effectif_total = EFFECTIF_FR + EFFECTIF_LUX

intensite = total_participations / effectif_total
score_engagement = min(100, intensite * 50)

print()
print(f"FAB\'Life (CACEIS pur)     : {FAB_LIFE_PARTICIPATIONS_CACEIS:>5}")
print(f"We Care (avec fallback)   : {we_care_total:>5}")
print(f"Be Generous (lauréats)    : {BE_GENEROUS_LAUREATS_CACEIS:>5}")
print(f"  Total participations    : {total_participations:>5}")
print(f"  Effectif (FR + Lux)     : {effectif_total:>5}")
print()
print(f"Intensité  = {total_participations} / {effectif_total} = {intensite:.3f} part./employé/an")
print(f"  → score_engagement = min(100, {intensite:.3f} × 50) = {score_engagement:.1f} / 100")

## 6. LSR final

In [ ]:
lsr = (score_mixite + score_inclusion + score_engagement) / 3

print(f"  Mixité      : {score_mixite:>5.1f}")
print(f"  Inclusion   : {score_inclusion:>5.1f}")
print(f"  Engagement  : {score_engagement:>5.1f}")
print(f"  -----------------------")
print(f"  LSR 2024    : {lsr:>5.1f} / 100")
print()

if lsr >= 75:
    band = "🟢 Zone verte"
elif lsr >= 60:
    band = "🟡 Zone jaune"
else:
    band = "🔴 Zone rouge"

print(f"Lecture board : {band}")

## 7. Sensibilité au fallback `n/a` de We Care

L'unique hypothèse du calcul est le **fallback de 24 participants** par ligne n/a. On vérifie ici que la conclusion exécutive ne dépend pas de cette hypothèse.

In [ ]:
sensibilite = []
for fallback in (0, 12, 24, 36, 48):
    we_care_var = WE_CARE_PARTICIPATIONS_CHIFFREES + WE_CARE_LIGNES_NA * fallback
    total_var   = FAB_LIFE_PARTICIPATIONS_CACEIS + we_care_var + BE_GENEROUS_LAUREATS_CACEIS
    intensite_var = total_var / effectif_total
    score_eng_var = min(100, intensite_var * 50)
    lsr_var = (score_mixite + score_inclusion + score_eng_var) / 3
    sensibilite.append({
        "Fallback n/a": fallback,
        "We Care total": we_care_var,
        "Engagement": round(score_eng_var, 1),
        "LSR": round(lsr_var, 1),
    })
sens_df = pd.DataFrame(sensibilite)
sens_df

**Lecture** : même en mettant 0 aux 14 lignes n/a (scénario impossible), le LSR ne perd que ~1 point. La conclusion exécutive *« zone jaune, à xx points du vert »* est **robuste** au choix de fallback.

## 8. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)

# ---- (a) Décomposition LSR ----
ax = axes[0]
labels = ["Mixité", "Inclusion", "Engagement", "LSR"]
values = [score_mixite, score_inclusion, score_engagement, lsr]
colors = [BLUE, STEEL, AMBER, RED]

bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=2, width=0.65)
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + 1.5, f"{v:.1f}",
            ha="center", fontsize=11, fontweight="bold", color=DARK)

ax.axhline(75, color=GREEN, linewidth=1, linestyle="--", alpha=0.6)
ax.text(3.55, 76, "Cible 75", color=GREEN, fontsize=8, ha="right")
ax.axhline(60, color=AMBER, linewidth=1, linestyle="--", alpha=0.5)
ax.text(3.55, 61, "Seuil 60", color=AMBER, fontsize=8, ha="right")

ax.set_ylim(0, 100)
ax.set_ylabel("Score (/100)", color=DARK)
ax.set_title("LSR 2024 — Décomposition", fontsize=13, loc="left", pad=10)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.spines["left"].set_color("#CFD6DD")
ax.spines["bottom"].set_color("#CFD6DD")
ax.grid(axis="y", color="#E5E9EE", linewidth=0.6)
ax.set_axisbelow(True)

# ---- (b) Sensibilité au fallback ----
ax = axes[1]
ax.fill_between(sens_df["Fallback n/a"], 60, 75, color=AMBER, alpha=0.10, label="Zone jaune")
ax.fill_between(sens_df["Fallback n/a"], 75, 100, color=GREEN, alpha=0.10, label="Zone verte")
ax.plot(sens_df["Fallback n/a"], sens_df["LSR"], marker="o", color=STEEL,
        linewidth=2.5, markersize=9, markerfacecolor="white", markeredgewidth=2)

# Marqueur retenu
fallback_retenu = WE_CARE_FALLBACK_PAR_NA
ax.axvline(fallback_retenu, color=RED, linewidth=1.5, linestyle="--", alpha=0.7)

ax.set_xlabel("Fallback (participations) appliqué aux 14 lignes n/a", color=DARK)
ax.set_ylabel("LSR final (/100)", color=DARK)
ax.set_title("Robustesse à l\'hypothèse fallback", fontsize=13, loc="left", pad=10)
ax.set_ylim(55, 75)
ax.legend(frameon=False, loc="lower right", fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#E5E9EE", linewidth=0.6)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lsr_2024_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.show()

## 9. Export CSV — compatible avec le dashboard Streamlit

Le LSR est exporté avec le même schéma de colonnes que les KPIs métier produits par `Code_Project.ipynb`, pour pouvoir être consommé par `app.py` sans changement du contrat de données.

In [ ]:
lsr_yearly = pd.DataFrame([{
    "KPI":            "LSR",
    "year":           2024,
    "scope":          "CACEIS Group (FR + Lux)",
    "Mixite":         round(score_mixite, 1),
    "Inclusion":      round(score_inclusion, 1),
    "Engagement":     round(score_engagement, 1),
    "LSR":            round(lsr, 1),
    "headcount":      effectif_total,
    "_paygap_pct":    round(gap_pct, 2),
    "_femmes_mgt":    round(pct_femmes_mgt, 2),
    "_intensite":     round(intensite, 3),
}])

out_path = OUTPUT_DIR / "lsr_yearly.csv"
# Convention du groupe (cf. KPIs_computations.ipynb cellule "Nettoyage final")
lsr_yearly.to_csv(out_path, index=False, sep=";", encoding="utf-8-sig")
print(f"→ {out_path.resolve()}")
lsr_yearly

## 10. Pitch board en 30 secondes

> *« CACEIS score 64/100 sur sa résilience sociale.*
>
> *Mixité quasi-cible (76 — la parité au management est atteinte à 39,75 %, le pay gap résiduel de 9,3 % est notre vrai levier).*
>
> *Inclusion en zone bonne (67 — Baromètre 2025 Mozaïk RH).*
>
> *Engagement à 49 — c'est notre plancher : avec ~0,99 participation par employé par an, on est juste sous la barre de 1, l'objectif minimum d'un programme actif. Quick-win HR : tracker les employés uniques pour passer d'une intensité à un vrai taux d'engagement.*
>
> *Zone jaune. Robustesse vérifiée : l'hypothèse de fallback sur les n/a de We Care fait varier le LSR final de moins de 1 point. »*

---

## 11. Références

| Source | Apport au calcul |
|---|---|
| `Bilan Social 2024.pdf` §2.1.2.2 | SAB moyen H/F par catégorie → pay gap |
| `Suivi accord mixité diversité QVT 2024 vDef.pdf` | Tableau cadres encadrants par classification × sexe → % femmes mgt |
| `Baromètre D&I CACEIS - France.pdf` §2.6 | Score sentiment d'inclusion FR (70 %) |
| `Baromètre D&I CACEIS - Luxembourg.pdf` §2.6 | Score sentiment d'inclusion Lux (64 %) |
| `Bilan Social 2024.pdf` + `Lux.pdf` §1.1.1 | Effectifs FR (2 043) + Lux (1 882) |
| `Bilan FAB'Life 2024.pptx` slide 31 | 2 984 participations totales (2 241 CACEIS pur) |
| `We Care - Bilan 2025.xlsx` sheet Data | 1 256 participations chiffrées + 14 lignes n/a |
| `Bilan Groupe Be Generous CACEIS 2025.xlsx` sheet Lauréats | 34 lauréats CACEIS (toutes entités) |

**Standards de référence :** ISO 30414 (Diversity · Health & Safety) · ESRS S1 (CSRD) · AI Act Art. 10 · Loi Rixain.

**Repository structure :**

```
.
├── Code_Project.ipynb              # 5 KPIs métier
├── LSR_KPI.ipynb                   # ce notebook
├── app.py                          # dashboard Streamlit
├── data/                           # source files (git-ignored)
├── outputs/
│   ├── kpi_yearly.csv
│   ├── kpi_by_entity.csv
│   ├── kpi_by_direction.csv
│   ├── lsr_yearly.csv              # ← produit par ce notebook
│   └── lsr_2024_dashboard.png
└── docs/LSR.md
```